# Chapter 9 — Assembling MAS with Subagents

## Setup Instructions

To ensure you have the required dependencies to run this notebook, you'll need to have our `llm-agents-from-scratch` framework installed on the running Jupyter kernel. To do this, you can launch this notebook with the following command while within the project's root directory:

```sh
uv run --with jupyter jupyter lab
```

Alternatively, if you just want to use the published version of `llm-agents-from-scratch` without local development, you can install it from PyPi by uncommenting the cell below.

In [ ]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPi
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code provided in this notebook, you'll need to have Ollama installed on your local machine and have its LLM hosting service running. To download Ollama, follow the instructions found on this page: https://ollama.com/download. After downloading and installing Ollama, you can start a service by opening a terminal and running the command `ollama serve`.

In [1]:
import os, shutil, subprocess, time, urllib.request, urllib.error


def ensure_ollama(host="http://localhost:11434", timeout=15):
    """Start Ollama if not already running and wait until responsive."""

    def _up():
        try:
            urllib.request.urlopen(f"{host}/api/tags", timeout=1)
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        return print(f"✓ Ollama already running at {host}")

    # Lightning persistent path first, then standard locations
    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        for candidate in [
            "/teamspace/studios/this_studio/.local/bin/ollama",
            "/usr/local/bin/ollama",
            "/usr/bin/ollama",
        ]:
            if os.path.exists(candidate):
                ollama_path = candidate
                break
    if ollama_path is None:
        raise RuntimeError(
            "Could not find the ollama binary. Install with: "
            "curl -fsSL https://ollama.com/install.sh | sh"
        )

    print(f"Starting Ollama server ({ollama_path})...")
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            return print(f"✓ Ollama up and running at {host}")
        time.sleep(0.5)

    raise RuntimeError(f"Ollama did not start within {timeout}s")


use_cloud = "OLLAMA_API_KEY" in os.environ
ensure_ollama() if not use_cloud else print("✓ Using Ollama Cloud")

✓ Using Ollama Cloud


## Examples

### Example 1: Manual Dispatch with UseSubAgentTool

Console logging is enabled below so you can watch the dispatched sub-agent run — its log lines are tagged `[hailstone]`, distinguishing them from the coordinator's own logs in later examples.

In [2]:
import logging

from llm_agents_from_scratch import LLMAgentBuilder
from llm_agents_from_scratch.data_structures import ToolCall
from llm_agents_from_scratch.llms import OllamaLLM
from llm_agents_from_scratch.logger import enable_console_logging
from llm_agents_from_scratch.subagents import SubAgentSpec, UseSubAgentTool
from llm_agents_from_scratch.tools import SimpleFunctionTool

enable_console_logging(logging.INFO)

model = "qwen3.5:397b-cloud" if use_cloud else "qwen3:14b"
host = "https://ollama.com" if use_cloud else None
llm = OllamaLLM(host=host, model=model, think=False, json_prompt_mode=use_cloud)


def next_number(x: int) -> int:
    if x % 2 == 0:
        return x // 2
    return 3 * x + 1


next_number_tool = SimpleFunctionTool(func=next_number)

spec = SubAgentSpec(
    name="hailstone",
    description="Computes Hailstone sequences using next_number.",
    builder=LLMAgentBuilder(llm=llm, tools=[next_number_tool]),
    max_steps=20,
)
tool = UseSubAgentTool(subagents_registry={spec.name: spec})

tool_call = ToolCall(
    tool_name="from_scratch__use_subagent",
    arguments={
        "name": "hailstone",
        "task": (
            "Compute the full Hailstone sequence for 5 step by step "
            "using next_number, until you reach 1. Report how many "
            "steps it took."
        ),
    },
)
result = await tool(tool_call=tool_call)
print(result.content)

[hailstone] INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Compute the full Hailstone sequence for 5 step by step using next_number, until you reach 1. Report how many steps it took.
[hailstone] INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Compute the full Hailstone sequence for 5 step by step using next_number, until you reach 1. Report how many steps it took.
[hailstone] INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: next_number
[hailstone] INFO (llm_agents_fs.TaskHandler) :      ✅ Successful Tool Call: 16
[hailstone] INFO (llm_agents_fs.TaskHandler) :      ✅ Step Result: I need to make the following tool-calls:
{
    "id_": "a5734e2c-a51c-48ea-bea0-47825f4a3cc2",
    "tool_name": "next_number",
    "argu...[TRUNCATED]
[hailstone] INFO (llm_agents_fs.TaskHandler) :      🧠 New Step: Execute the tool call with id 'a5734e2c-a51c-48ea-bea0-47825f4a3cc2' using the next_number tool with argument x=16.
[hailstone] INFO (llm_agents_fs.TaskHandler) :    

### Example 2: Automatic Dispatch via the Coordinator

In [3]:
from llm_agents_from_scratch.data_structures import Task

# reuses `spec` from Example 1 -- the same hailstone SubAgentSpec, no
# changes needed. SubAgentSpec.builder builds a fresh LLMAgent per
# dispatch, so it's safe to reuse across coordinators.
hailstone_coordinator = await (
    LLMAgentBuilder().with_llm(llm).with_subagents([spec]).build()
)

task_hailstone = Task(
    instruction="Determine the Hailstone sequence for the number 5.",
)
result_hailstone = await hailstone_coordinator.run(task_hailstone)
print(result_hailstone.content)

INFO

 (llm_agents_fs.LLMAgent) :      🚀 Starting task: Determine the Hailstone sequence for the number 5.


INFO

 (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Determine the Hailstone sequence for the number 5.


INFO

...

INFO

 (llm_agents_fs.TaskHandler) :      No new step required.


INFO

 (llm_agents_fs.LLMAgent) :      🏁 Task completed: The Hailstone sequence for the number 5 has been successfully determined. Here is the sequence:

**Sequence:** 5 → 16 → 8 → 4 → 2 → ...[TRUNCATED]


The Hailstone sequence for the number 5 has been successfully determined. Here is the sequence:

**Sequence:** 5 → 16 → 8 → 4 → 2 → 1  
**Starting number:** 5  
**Total steps taken:** 5  
**Maximum value reached:** 16  

Let me know if you'd like to explore further!

### Example 3: HITL on the Coordinator

In [4]:
from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.tools.default import SharedConsoleHumanInputTool

# both subagents here do simple, tightly-scoped work, so a small
# local model is plenty
slm = OllamaLLM(model="qwen3:8b", think=False)

# two subagents, each with its own SharedConsoleHumanInputTool instance
# -- but the lock is a CLASS-level asyncio.Lock, so both instances
# share it. Dispatched concurrently, their prompts serialize instead
# of racing for stdin.
hailstone_input_tool = SharedConsoleHumanInputTool(agent_name="hailstone")
hailstone_spec = SubAgentSpec(
    name="hailstone",
    description=(
        "Asks the human operator for a starting number, then computes "
        "its Hailstone sequence."
    ),
    builder=LLMAgentBuilder(
        llm=slm,
        tools=[next_number_tool, hailstone_input_tool],
    ),
    max_steps=20,
)

greeter_input_tool = SharedConsoleHumanInputTool(agent_name="greeter")
greeter_spec = SubAgentSpec(
    name="greeter",
    description="Asks the human operator for their name, then greets them.",
    builder=LLMAgentBuilder(llm=slm, tools=[greeter_input_tool]),
    max_steps=5,
)

# the builder's only async work is MCP tool discovery, and this
# coordinator has no MCP providers, so construct the LLMAgent directly.
# The subagent specs above still use builders: SubAgentSpec.builder is a
# recipe, rebuilt fresh on every dispatch.
hitl_coordinator = LLMAgent(
    llm=llm,
    subagents=[hailstone_spec, greeter_spec],
)

task_hitl = Task(
    instruction=(
        "Dispatch both the hailstone and greeter subagents in the same "
        "response so they run concurrently. hailstone should ask the "
        "human for a starting number, then compute its Hailstone "
        "sequence. greeter should ask the human for their name, then "
        "return a friendly greeting. Report both results."
    ),
)
# watch the console: the two prompts complete one at a time, never
# interleaved, thanks to SharedConsoleHumanInputTool's shared lock.
result_hitl = await hitl_coordinator.run(task_hitl)
print(result_hitl.content)

INFO

 (llm_agents_fs.LLMAgent) :      🚀 Starting task: Dispatch both the hailstone and greeter subagents in the same response so they run concurrently. hailstone should ask the human for a...[TRUNCATED]


INFO

 (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Dispatch both the hailstone and greeter subagents in the same response so they run concurrently. hailstone should ask the human fo...[TRUNCATED]


INFO

...

INFO

 (llm_agents_fs.TaskHandler) :      No new step required.


INFO

 (llm_agents_fs.LLMAgent) :      🏁 Task completed: Both subagents have completed their tasks. Let me summarize the results:

**Hailstone Subagent Results:**
- Starting number: 4
- Hai...[TRUNCATED]


Both subagents have completed their tasks. Let me summarize the results:

**Hailstone Subagent Results:**
- Starting number: 4
- Hailstone sequence: 4 → 2 → 1
- Total steps: 2
- Maximum value reached: 4

**Greeter Subagent Results:**
- The human's name is Andrei
- A friendly greeting was returned using that name

Both subagents ran concurrently and completed successfully. The hailstone sequence for starting number 4 reached 1 in just 2 steps, and the greeter learned the user's name is Andrei and provided a friendly greeting.

### Example 4: Router/Triage

In [5]:
from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.subagents.recipes import explore_subagent, general_subagent

# explore only does lookups, so the small model from Example 3 is
# plenty; general does open-ended computation, so it shares the
# coordinator's own model
coordinator = LLMAgent(
    llm=llm,  # the bigger model from Example 1
    subagents=[general_subagent(llm), explore_subagent(slm)],
)

# a lookup task — should route to explore (reads hailstone_known_sequences.json)
task_lookup = Task(
    instruction=(
        "Look up the Hailstone sequence for 6 in "
        "hailstone_known_sequences.json and report it."
    ),
)
result_lookup = await coordinator.run(task_lookup)
print(result_lookup.content)

INFO

 (llm_agents_fs.LLMAgent) :      🚀 Starting task: Look up the Hailstone sequence for 6 in hailstone_known_sequences.json and report it.


INFO

 (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Look up the Hailstone sequence for 6 in hailstone_known_sequences.json and report it.


INFO

...

INFO

 (llm_agents_fs.TaskHandler) :      No new step required.


INFO

 (llm_agents_fs.LLMAgent) :      🏁 Task completed: The Hailstone sequence for 6 is: **[6, 3, 10, 5, 16, 8, 4, 2, 1]**

It takes 8 steps to reach the number 1.


The Hailstone sequence for 6 is: **[6, 3, 10, 5, 16, 8, 4, 2, 1]**

It takes 8 steps to reach the number 1.

### Example 5: Parallel Fan-Out

In [6]:
task_fanout = Task(
    instruction=(
        "Dispatch three sub-agent calls in the same response so they run "
        "concurrently: use general to compute the Hailstone sequence for "
        "4, use general to compute it for 8, and use explore to look up "
        "the sequence for 12 in hailstone_known_sequences.json. Report "
        "which of the three starting numbers took the most steps."
    ),
)
# gather() runs the three dispatches concurrently, but wall-clock
# speedup isn't guaranteed: a single local Ollama instance queues
# concurrent requests to the *same* model unless OLLAMA_NUM_PARALLEL is
# raised. Cloud models parallelize for free.
result_fanout = await coordinator.run(task_fanout)
print(result_fanout.content)

INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Dispatch three sub-agent calls in the same response so they run concurrently: use general to compute the Hailstone sequence for 4, us...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Dispatch three sub-agent calls in the same response so they run concurrently: use general to compute the Hailstone sequence for 4,...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: from_scratch__use_subagent
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: from_scratch__use_subagent
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: from_scratch__use_subagent
[general] INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Compute the Hailstone sequence for starting number 4. The Hailstone sequence (also known as Collatz sequence) is generated by: if the...[TRUNCATED]
[general] INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Compute the Hailstone sequence for sta

### Example 6: Sequence

In [7]:
# A chain assembled entirely out of specs we already have. `explore`
# (Example 4) is given ReadFileTool and no next_number, so it can read
# the file but has no way to compute a sequence; `spec` (Example 1's
# hailstone) is given next_number_tool and no ReadFileTool, so it can
# compute but has no way to read the file. Neither can answer alone.
#
# The Task instruction below is of course written up front. What cannot
# be written up front is the *dispatch* the coordinator sends to
# hailstone: the starting number inside it does not exist until explore
# has answered and the coordinator has done arithmetic on that answer.
# That is what makes this a sequence rather than the fan-out above,
# where all three dispatches were fully determined before any of them
# ran.
#
# Note that `explore` gets the coordinator's own model here, not the
# small one it was given in Example 4. The difference is what the lookup
# feeds. There, a garbled read just produces a bad answer the reader can
# see. Here the value read out of the file gets doubled and written into
# a second dispatch, so an unreliable lookup silently corrupts every
# step downstream of it. In a chain, the first link has to be trustworthy.
sequence_coordinator = await (
    LLMAgentBuilder()
    .with_llm(llm)
    .with_subagents([spec, explore_subagent(llm)])
    .build()
)

# `explore` is asked only to read and report, never to compare. Picking
# the largest step count and doubling it are both left to the
# coordinator, matching what explore_subagent() is built for: "lookups
# and fact-finding, not multi-step work".
task_sequence = Task(
    instruction=(
        "Ask explore to read hailstone_known_sequences.json and report "
        "its contents. Compare the recorded step counts yourself and "
        "take the starting number with the most steps. Then ask "
        "hailstone to compute the full Hailstone sequence for double "
        "that number. Report the sequence and whether it took more "
        "steps than the recorded entry."
    ),
)
sequence_handler = sequence_coordinator.run(task_sequence, max_steps=10)
result_sequence = await sequence_handler
print(result_sequence.content)

INFO

 (llm_agents_fs.LLMAgent) :      🚀 Starting task: Ask explore to read hailstone_known_sequences.json and report its contents. Compare the recorded step counts yourself and take the st...[TRUNCATED]


INFO

 (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Ask explore to read hailstone_known_sequences.json and report its contents. Compare the recorded step counts yourself and take the...[TRUNCATED]


INFO

...

INFO

 (llm_agents_fs.TaskHandler) :      No new step required.


INFO

 (llm_agents_fs.LLMAgent) :      🏁 Task completed: Now I have all the information needed to complete the task. Let me analyze and compare:

**From hailstone_known_sequences.json:**
- ...[TRUNCATED]


Now I have all the information needed to complete the task. Let me analyze and compare:

**From hailstone_known_sequences.json:**
- Starting number 6: 8 steps
- Starting number 12: 9 steps

The starting number with the most steps is **12** with **9 steps**.

**Double that number:** 12 × 2 = **24**

**Hailstone sequence for 24:**
- Sequence: 24 → 12 → 6 → 3 → 10 → 5 → 16 → 8 → 4 → 2 → 1
- Steps: **10 steps**

**Comparison:**
- Recorded entry for 12: 9 steps
- Computed sequence for 24: 10 steps

**Answer:** Yes, the Hailstone sequence for 24 took **more steps** (10 steps) than the recorded entry for 12 (9 steps). It took 1 additional step.

`sequence_handler.rollout` is what shows this really was a chain. `explore` reports what the file contains, then the coordinator picks the entry with the most steps, doubles it, and writes the result into `hailstone`'s dispatch instruction. That second dispatch could not have been issued until the first one answered, and the number in it appears nowhere in the original task. The coordinator does not just relay a value here, it transforms one.

The doubling also gives us a free correctness check. Any even number `2n` steps straight down to `n`, so a doubled starting number always takes exactly one more step than the original.

In [8]:
print(sequence_handler.rollout)

=== Task Step Start ===

💬 assistant: My current instruction is 'Ask explore to read hailstone_known_sequences.json and report its contents. Compare the recorded step counts yourself and take the starting number with the most steps. Then ask hailstone to compute the full Hailstone sequence for double that number. Report the sequence and whether it took more steps than the recorded entry.'

💬 assistant: I need to make the following tool call(s):

{
    "id_": "8b27f37e-f35b-4236-a34d-8dd23d6eebc8",
    "tool_name": "from_scratch__use_subagent",
    "arguments": {
        "name": "explore",
        "task": "Read hailstone_known_sequences.json and report its contents."
    }
}.

🔧 tool: {
    "tool_call_id": "8b27f37e-f35b-4236-a34d-8dd23d6eebc8",
    "content": "The file `hailstone_known_sequences.json` contains data about hailstone sequences (also known as Collatz sequences) for two starting numbers:\n\n**Contents:**\n\n1. **Starting number 6:**\n   - Sequence: [6, 3, 10, 5, 16, 8, 4, 2